# Agent Loop 停滞检测：状态指纹、无进展预算与安全终止

**面试问题：Agent 反复调用同一工具却没有进展时，怎样检测并停止，而不误伤正常轮询？**

## 回答主线

1. 只设置最大步数能最终停机，却会在错误动作上浪费大量工具调用和费用。
2. 停滞指纹应包含规范化意图、动作、关键参数、环境状态哈希和错误类别。
3. 连续重复同一指纹或状态长期不变说明没有信息增益，可以触发改写、换工具、询问用户或停止。
4. 预算策略要同时限制步数、工具成本、墙钟时间和相同错误次数。
5. 正常轮询会重复动作，但观察版本或状态会变化，且应遵守退避和明确等待期限。
6. 终止事件必须保留最后证据和可恢复下一步，而不是只返回“失败”。

## 真实案例

五条 Agent 轨迹覆盖无效订单号循环、权限拒绝循环、分页游标前进、支付状态轮询和搜索改写。每条包含动作、参数、观察版本、状态与成本。我们比较固定 10 步上限与指纹检测，并构造正常 polling 被朴素重复规则误杀的反例。这是可读的离线教学实验，用来验证协议和算法，不能外推为线上模型收益。

### 输入预览：五类轨迹事件

In [1]:
import hashlib  # 导入摘要算法以生成规范化状态指纹。
import json  # 导入稳定序列化以处理动作参数。

traces = [  # 构造五条具有不同进展语义的 Agent 轨迹。
    {"id": "L1", "kind": "invalid-order", "events": [{"action": "order.get", "args": {"id": "BAD"}, "state": "not_found", "obs_version": 1, "cost": 1} for _ in range(6)]},  # 完全相同的无效订单循环。
    {"id": "L2", "kind": "permission", "events": [{"action": "refund.create", "args": {"amount": 899}, "state": "permission_denied", "obs_version": 1, "cost": 3} for _ in range(5)]},  # 权限拒绝后盲目重试。
    {"id": "L3", "kind": "pagination", "events": [{"action": "search.next", "args": {"cursor": cursor}, "state": f"page_{index}", "obs_version": index, "cost": 1} for index, cursor in enumerate(["c1", "c2", "c3", "c4"], start=1)]},  # 动作相似但游标和状态持续前进。
    {"id": "L4", "kind": "polling", "events": [{"action": "payment.status", "args": {"id": "P7"}, "state": state, "obs_version": index, "cost": 1} for index, state in enumerate(["pending", "pending", "processing", "paid"], start=1)]},  # 正常外部状态轮询。
    {"id": "L5", "kind": "rewrite", "events": [{"action": "search", "args": {"query": query}, "state": state, "obs_version": index, "cost": 1} for index, (query, state) in enumerate([("退款", "zero_hits"), ("退款 到账 时间", "two_hits"), ("退款 3天", "answer_found")], start=1)]},  # 查询逐步改写并获得信息。
]  # 完成轨迹集合。
print("轨迹  类型          steps  states")  # 输出输入表头。
for trace in traces:  # 逐轨迹展示状态序列。
    print(f"{trace['id']}   {trace['kind']:<13} {len(trace['events']):>5}  {[event['state'] for event in trace['events']]}")  # 对比循环和真实进展。

轨迹  类型          steps  states
L1   invalid-order     6  ['not_found', 'not_found', 'not_found', 'not_found', 'not_found', 'not_found']
L2   permission        5  ['permission_denied', 'permission_denied', 'permission_denied', 'permission_denied', 'permission_denied']
L3   pagination        4  ['page_1', 'page_2', 'page_3', 'page_4']
L4   polling           4  ['pending', 'pending', 'processing', 'paid']
L5   rewrite           3  ['zero_hits', 'two_hits', 'answer_found']


## Baseline 基线：统一跑满 10 步才停止

In [2]:
def max_step_baseline(trace, max_steps=10):  # 模拟只依赖固定步数的停机策略。
    template = trace["events"]  # 读取已有轨迹模板。
    executed = []  # 收集实际执行事件。
    for step in range(max_steps):  # 不检查进展地执行到上限。
        event = template[min(step, len(template) - 1)]  # 轨迹结束后重复最后一次错误动作。
        executed.append(event)  # 记录当前工具调用。
        if event["state"] in {"paid", "answer_found", "page_4"}:  # 明确成功状态可以提前结束。
            break  # 正常目标已完成。
    return executed  # 返回实际调用序列。

baseline_rows = []  # 收集五条轨迹的基线成本。
for trace in traces:  # 逐轨迹运行最大步数策略。
    executed = max_step_baseline(trace)  # 获取执行事件。
    baseline_rows.append({"id": trace["id"], "steps": len(executed), "cost": sum(event["cost"] for event in executed), "final": executed[-1]["state"]})  # 保存步数、成本和终态。
print("轨迹  baseline_steps  cost  final")  # 输出基线结果表头。
for row in baseline_rows:  # 逐轨迹展示无效循环成本。
    print(f"{row['id']} {row['steps']:>15} {row['cost']:>5}  {row['final']}")  # 展示 L1/L2 跑满十步。

轨迹  baseline_steps  cost  final
L1              10    10  not_found
L2              10    30  permission_denied
L3               4     4  page_4
L4               4     4  paid
L5               3     3  answer_found


### 核心实现：规范化指纹、进展信号与多预算

In [3]:
def event_fingerprint(event, include_observation=True):  # 为一次动作和环境状态生成可比较指纹。
    protected = {"action": event["action"], "args": event["args"], "state": event["state"]}  # 绑定动作、参数和语义状态。
    if include_observation:  # 正常轮询需要区分环境观察版本。
        protected["obs_version"] = event["obs_version"]  # 加入单调变化的观察版本。
    canonical = json.dumps(protected, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 生成稳定 JSON。
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]  # 返回短指纹供事件账本使用。

def run_with_stagnation_policy(trace, repeat_limit=3, max_steps=10, max_cost=12):  # 综合重复、步数和成本执行轨迹。
    executed = []  # 收集已执行事件。
    fingerprint_counts = {}  # 统计相同动作状态指纹出现次数。
    stop_reason = "trace-ended"  # 默认按样本轨迹自然结束。
    for event in trace["events"]:  # 按 Agent 产生的事件顺序执行。
        if len(executed) >= max_steps or sum(item["cost"] for item in executed) + event["cost"] > max_cost:  # 在副作用前检查硬预算。
            stop_reason = "hard-budget"  # 记录步数或成本上限。
            break  # 停止继续调用工具。
        fingerprint = event_fingerprint(event, include_observation=False)  # 用语义状态检测真正相同的无进展动作。
        fingerprint_counts[fingerprint] = fingerprint_counts.get(fingerprint, 0) + 1  # 累加重复次数。
        executed.append({**event, "fingerprint": fingerprint})  # 保存带指纹的事件。
        if fingerprint_counts[fingerprint] >= repeat_limit:  # 同一动作、参数和状态连续/累计达到门槛。
            stop_reason = "stagnation-repeat"  # 标记停滞终止。
            break  # 避免继续浪费调用。
        if event["state"] in {"paid", "answer_found", "page_4"}:  # 检查任务成功状态。
            stop_reason = "completed"  # 标记正常完成。
            break  # 结束轨迹。
    return executed, stop_reason  # 返回执行账本和终止原因。

demo_events, demo_reason = run_with_stagnation_policy(traces[0])  # 对无效订单循环演示检测。
print("L1 指纹账本：")  # 输出重复指纹的中间过程。
for step, event in enumerate(demo_events, start=1):  # 逐步展示相同状态哈希。
    print(f"step={step} action={event['action']} state={event['state']} fp={event['fingerprint']}")  # 展示第三次触发停滞。
print("停止原因：", demo_reason)  # 展示不是等到十步上限。

L1 指纹账本：
step=1 action=order.get state=not_found fp=96f7cdf91a75
step=2 action=order.get state=not_found fp=96f7cdf91a75
step=3 action=order.get state=not_found fp=96f7cdf91a75
停止原因： stagnation-repeat


## 结果解读：节省调用且保留真正进展

In [4]:
policy_rows = []  # 收集五条轨迹的新策略结果。
for trace in traces:  # 逐轨迹运行停滞检测。
    executed, reason = run_with_stagnation_policy(trace)  # 获取实际事件和终止原因。
    policy_rows.append({"id": trace["id"], "steps": len(executed), "cost": sum(event["cost"] for event in executed), "final": executed[-1]["state"], "reason": reason})  # 保存同口径指标。
print("轨迹  Baseline步  Policy步  Baseline成本  Policy成本  reason")  # 输出策略对照表头。
for baseline, policy in zip(baseline_rows, policy_rows):  # 对齐同一轨迹两种执行方式。
    print(f"{policy['id']} {baseline['steps']:>10} {policy['steps']:>9} {baseline['cost']:>13} {policy['cost']:>11}  {policy['reason']}")  # 展示循环节省和成功轨迹保留。
saved_calls = sum(row["steps"] for row in baseline_rows) - sum(row["steps"] for row in policy_rows)  # 汇总减少的工具调用。
completed = [row["id"] for row in policy_rows if row["reason"] == "completed"]  # 收集正常完成轨迹。
print(f"减少工具调用={saved_calls}，正常完成={completed}")  # 同时报告效率和任务完整性。
print("解读：L1/L2 在第三次同状态时停止；分页、轮询和查询改写因为参数或状态变化继续到成功。")  # 解释指纹维度的作用。

轨迹  Baseline步  Policy步  Baseline成本  Policy成本  reason
L1         10         3            10           3  stagnation-repeat
L2         10         3            30           9  stagnation-repeat
L3          4         4             4           4  completed
L4          4         4             4           4  completed
L5          3         3             3           3  completed
减少工具调用=14，正常完成=['L3', 'L4', 'L5']
解读：L1/L2 在第三次同状态时停止；分页、轮询和查询改写因为参数或状态变化继续到成功。


## 失败案例：只看 action 名称会误杀正常 Payment Polling

In [5]:
def action_only_guard(trace, repeat_limit=2):  # 实现只统计工具名称的过度简化检测器。
    counts = {}  # 统计每个 action 出现次数。
    executed = []  # 收集检测器允许的事件。
    for event in trace["events"]:  # 遍历支付状态轮询。
        counts[event["action"]] = counts.get(event["action"], 0) + 1  # 忽略观察版本和状态变化。
        executed.append(event)  # 记录当前轮询。
        if counts[event["action"]] >= repeat_limit:  # 第二次相同工具名就停止。
            return executed, "false-stagnation"  # 在支付仍 pending 时误杀。
    return executed, "completed"  # 返回自然结束。

naive_poll_events, naive_poll_reason = action_only_guard(traces[3])  # 对正常轮询运行朴素规则。
safe_poll_events, safe_poll_reason = run_with_stagnation_policy(traces[3])  # 用状态指纹运行正常轮询。
print(f"action-only：steps={len(naive_poll_events)} final={naive_poll_events[-1]['state']} reason={naive_poll_reason}")  # 展示在第二次 pending 停止。
print(f"state-aware：steps={len(safe_poll_events)} final={safe_poll_events[-1]['state']} reason={safe_poll_reason}")  # 展示观察版本和状态推进到 paid。
print("修正策略：轮询合同包含 observation_version、deadline 和退避；只有同版本同状态重复才算无进展，不能只数工具名。")  # 总结防误杀规则。

action-only：steps=2 final=pending reason=false-stagnation
state-aware：steps=4 final=paid reason=completed
修正策略：轮询合同包含 observation_version、deadline 和退避；只有同版本同状态重复才算无进展，不能只数工具名。


### 生产边界与停止事件

In [6]:
stop_event = {"trace_id": "L2", "reason": policy_rows[1]["reason"], "steps": policy_rows[1]["steps"], "cost": policy_rows[1]["cost"], "last_state": policy_rows[1]["final"], "recovery": "request-human-approval", "policy_version": "loop-stop-r3"}  # 构造可恢复停止事件。
print("停止事件：", stop_event)  # 展示终止后仍提供下一步。
print("生产替换点：真实系统还需跨 span 状态摘要、语义动作归一化、墙钟/Token 预算、分布式取消、外部状态版本和人工接管。")  # 明确内存轨迹边界。

停止事件： {'trace_id': 'L2', 'reason': 'stagnation-repeat', 'steps': 3, 'cost': 9, 'last_state': 'permission_denied', 'recovery': 'request-human-approval', 'policy_version': 'loop-stop-r3'}
生产替换点：真实系统还需跨 span 状态摘要、语义动作归一化、墙钟/Token 预算、分布式取消、外部状态版本和人工接管。


## 回归测试：最后只保护循环终止、进展保留与轮询反例

In [7]:
assert policy_rows[0]["steps"] == 3 and policy_rows[0]["reason"] == "stagnation-repeat"  # 验证无效订单在第三次重复时停止。
assert policy_rows[1]["steps"] == 3 and policy_rows[1]["cost"] < baseline_rows[1]["cost"]  # 验证昂贵权限拒绝循环减少成本。
assert {"L3", "L4", "L5"}.issubset(set(completed))  # 验证分页、轮询和改写轨迹正常完成。
assert saved_calls > 0  # 验证策略在不牺牲成功轨迹时减少工具调用。
assert naive_poll_reason == "false-stagnation" and safe_poll_reason == "completed" and safe_poll_events[-1]["state"] == "paid"  # 验证 action-only 误杀和状态感知修正。
print("回归测试通过：三次停滞、成本节省、三类进展完成、调用下降和轮询误杀修正均成立。")  # 用少量断言总结 Loop 合同。

回归测试通过：三次停滞、成本节省、三类进展完成、调用下降和轮询误杀修正均成立。
